In [ ]:
import pandas as pd
import numpy as np
from numpy import genfromtxt
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.gridspec as gridspec
import seaborn
import scipy
import scipy.stats as st
import scipy.io
from sklearn.neighbors import KernelDensity
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
from dPCA import dPCA
import string
import warnings
import pickle
import json

from scipy.ndimage import gaussian_filter1d

import sys
import os
import glob
import random
from time import time


from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests

from scipy.ndimage import label


In [ ]:
# to be able to use the functions in the ana_functions under /3d_recontruction_analysis_self_and_coop_task_neural_analysis/
sys.path.append(os.path.abspath('../3d_recontruction_analysis_self_and_coop_task_neural_analysis/'))

In [ ]:
from ana_functions.body_part_locs_eachpair import body_part_locs_eachpair
from ana_functions.body_part_locs_singlecam import body_part_locs_singlecam

from ana_functions.bhv_events_timepoint import bhv_events_timepoint
from ana_functions.bhv_events_timepoint_singlecam import bhv_events_timepoint_singlecam

## Analyze each session

In [ ]:
# instead of using gaze angle threshold, use the target rectagon to deside gaze info
# ...need to update
sqr_thres_tubelever = 75 # draw the square around tube and lever
sqr_thres_face = 1.15 # a ratio for defining face boundary
sqr_thres_body = 4 # how many times to enlongate the face box boundry to the body


# get the fps of the analyzed video
fps = 30

# get the fs for neural recording
fs_spikes = 20000
fs_lfp = 1000

# frame number of the demo video
nframes = 0.5*30 # second*30fps
# nframes = 45*30 # second*30fps

# re-analyze the video or not
reanalyze_video = 0
redo_anystep = 0

# if use onset of the first increase after min
doOnsetAfterMin = 0
if not doOnsetAfterMin:
    doOnsetAfterMin_suffix = ''
elif doOnsetAfterMin:
    doOnsetAfterMin_suffix = 'PullOnsetAfterMin_'

# do OFC sessions or DLPFC sessions
do_OFC = 0
do_DLPFC  = 1
if do_OFC:
    savefile_sufix = '_OFCs'
elif do_DLPFC:
    savefile_sufix = '_DLPFCs'
else:
    savefile_sufix = ''
    
# all the videos (no misaligned ones)
# aligned with the audio
# get the session start time from "videosound_bhv_sync.py/.ipynb"
# currently the session_start_time will be manually typed in. It can be updated after a better method is used


# dodson ginger
if 1:
    if do_DLPFC:
        neural_record_conditions = [
                                    '20240531_Dodson_MC',
                                    '20240603_Dodson_MC_and_SR',
                                    '20240603_Dodson_MC_and_SR',
                                    '20240604_Dodson_MC',
                                    '20240605_Dodson_MC_and_SR',
                                    '20240605_Dodson_MC_and_SR',
                                    '20240606_Dodson_MC_and_SR',
                                    '20240606_Dodson_MC_and_SR',
                                    '20240607_Dodson_SR',
                                    '20240610_Dodson_MC',
                                    '20240611_Dodson_SR',
                                    '20240612_Dodson_MC',
                                    '20240613_Dodson_SR',
                                    '20240620_Dodson_SR',
                                    '20240719_Dodson_MC',
                                        
                                    '20250129_Dodson_MC',
                                    '20250130_Dodson_SR',
                                    '20250131_Dodson_MC',
                                
            
                                    '20250210_Dodson_SR_withKoala',
                                    '20250211_Dodson_MC_withKoala',
                                    '20250212_Dodson_SR_withKoala',
                                    '20250214_Dodson_MC_withKoala',
                                    '20250217_Dodson_SR_withKoala',
                                    '20250218_Dodson_MC_withKoala',
                                    '20250219_Dodson_SR_withKoala',
                                    '20250220_Dodson_MC_withKoala',
                                    '20250224_Dodson_KoalaAL_withKoala',
                                    '20250226_Dodson_MC_withKoala',
                                    '20250227_Dodson_KoalaAL_withKoala',
                                    '20250228_Dodson_DodsonAL_withKoala',
                                    '20250304_Dodson_DodsonAL_withKoala',
                                    '20250305_Dodson_MC_withKoala',
                                    '20250306_Dodson_KoalaAL_withKoala',
                                    '20250307_Dodson_DodsonAL_withKoala',
                                    '20250310_Dodson_MC_withKoala',
                                    '20250312_Dodson_NV_withKoala',
                                    '20250313_Dodson_NV_withKoala',
                                    '20250314_Dodson_NV_withKoala',
            
                                    '20250401_Dodson_MC_withKanga',
                                    '20250402_Dodson_MC_withKanga',
                                    '20250403_Dodson_MC_withKanga',
                                    '20250404_Dodson_SR_withKanga',
                                    '20250407_Dodson_SR_withKanga',
                                    '20250408_Dodson_SR_withKanga',
                                    '20250409_Dodson_MC_withKanga',
            
                                    '20250415_Dodson_MC_withKanga',
                                    # '20250416_Dodson_SR_withKanga', # has to remove from the later analysis, recording has problems
                                    '20250417_Dodson_MC_withKanga',
                                    '20250418_Dodson_SR_withKanga',
                                    '20250421_Dodson_SR_withKanga',
                                    '20250422_Dodson_MC_withKanga',
                                    '20250422_Dodson_SR_withKanga',
            
                                    '20250423_Dodson_MC_withKanga',
                                    '20250423_Dodson_SR_withKanga', 
                                    '20250424_Dodson_NV_withKanga',
                                    '20250424_Dodson_MC_withKanga',
                                    '20250424_Dodson_SR_withKanga',            
                                    '20250425_Dodson_NV_withKanga',
                                    '20250425_Dodson_SR_withKanga',
                                    '20250428_Dodson_NV_withKanga',
                                    '20250428_Dodson_MC_withKanga',
                                    '20250428_Dodson_SR_withKanga',  
                                    '20250429_Dodson_NV_withKanga',
                                    '20250429_Dodson_MC_withKanga',
                                    '20250429_Dodson_SR_withKanga',  
                                    '20250430_Dodson_NV_withKanga',
                                    '20250430_Dodson_MC_withKanga',
                                    '20250430_Dodson_SR_withKanga',  
            
                                   ]
        task_conditions = [
                            'MC',           
                            'MC',
                            'SR',
                            'MC',
                            'MC',
                            'SR',
                            'MC',
                            'SR',
                            'SR',
                            'MC',
                            'SR',
                            'MC',
                            'SR',
                            'SR',
                            'MC',
                            
                            'MC_withGingerNew',
                            'SR_withGingerNew',
                            'MC_withGingerNew',
            
                            'SR_withKoala',
                            'MC_withKoala',
                            'SR_withKoala',
                            'MC_withKoala',
                            'SR_withKoala',
                            'MC_withKoala',
                            'SR_withKoala',
                            'MC_withKoala',
                            'MC_KoalaAuto_withKoala',
                            'MC_withKoala',
                            'MC_KoalaAuto_withKoala',
                            'MC_DodsonAuto_withKoala',
                            'MC_DodsonAuto_withKoala',
                            'MC_withKoala',
                            'MC_KoalaAuto_withKoala',
                            'MC_DodsonAuto_withKoala',
                            'MC_withKoala',
                            'NV_withKoala',
                            'NV_withKoala',
                            'NV_withKoala',

                            'MC_withKanga',
                            'MC_withKanga',
                            'MC_withKanga',
                            'SR_withKanga',
                            'SR_withKanga',
                            'SR_withKanga',
                            'MC_withKanga',
            
                            'MC_withKanga',
                            # 'SR_withKanga',
                            'MC_withKanga',
                            'SR_withKanga',
                            'SR_withKanga',
                            'MC_withKanga',
                            'SR_withKanga',
            
                            'MC_withKanga',
                            'SR_withKanga', 
                            'NV_withKanga',
                            'MC_withKanga',
                            'SR_withKanga',            
                            'NV_withKanga',
                            'SR_withKanga',
                            'NV_withKanga',
                            'MC_withKanga',
                            'SR_withKanga',  
                            'NV_withKanga',
                            'MC_withKanga',
                            'SR_withKanga',  
                            'NV_withKanga',
                            'MC_withKanga',
                            'SR_withKanga',  
                          ]
        dates_list = [
                        '20240531',
                        '20240603_MC',
                        '20240603_SR',
                        '20240604',
                        '20240605_MC',
                        '20240605_SR',
                        '20240606_MC',
                        '20240606_SR',
                        '20240607',
                        '20240610_MC',
                        '20240611',
                        '20240612',
                        '20240613',
                        '20240620',
                        '20240719',
            
                        '20250129',
                        '20250130',
                        '20250131',
            
                        '20250210',
                        '20250211',
                        '20250212',
                        '20250214',
                        '20250217',
                        '20250218',
                        '20250219',
                        '20250220',
                        '20250224',
                        '20250226',
                        '20250227',
                        '20250228',
                        '20250304',
                        '20250305',
                        '20250306',
                        '20250307',
                        '20250310',
                        '20250312',
                        '20250313',
                        '20250314',
            
                        '20250401',
                        '20250402',
                        '20250403',
                        '20250404',
                        '20250407',
                        '20250408',
                        '20250409',
            
                        '20250415',
                        # '20250416',
                        '20250417',
                        '20250418',
                        '20250421',
                        '20250422',
                        '20250422_SR',
            
                        '20250423',
                        '20250423_SR', 
                        '20250424',
                        '20250424_MC',
                        '20250424_SR',            
                        '20250425',
                        '20250425_SR',
                        '20250428_NV',
                        '20250428_MC',
                        '20250428_SR',  
                        '20250429_NV',
                        '20250429_MC',
                        '20250429_SR',  
                        '20250430_NV',
                        '20250430_MC',
                        '20250430_SR',  
                     ]
        videodates_list = [
                            '20240531',
                            '20240603',
                            '20240603',
                            '20240604',
                            '20240605',
                            '20240605',
                            '20240606',
                            '20240606',
                            '20240607',
                            '20240610_MC',
                            '20240611',
                            '20240612',
                            '20240613',
                            '20240620',
                            '20240719',
            
                            '20250129',
                            '20250130',
                            '20250131',
                            
                            '20250210',
                            '20250211',
                            '20250212',
                            '20250214',
                            '20250217',
                            '20250218',          
                            '20250219',
                            '20250220',
                            '20250224',
                            '20250226',
                            '20250227',
                            '20250228',
                            '20250304',
                            '20250305',
                            '20250306',
                            '20250307',
                            '20250310',
                            '20250312',
                            '20250313',
                            '20250314',
            
                            '20250401',
                            '20250402',
                            '20250403',
                            '20250404',
                            '20250407',
                            '20250408',
                            '20250409',
            
                            '20250415',
                            # '20250416',
                            '20250417',
                            '20250418',
                            '20250421',
                            '20250422',
                            '20250422_SR',
            
                            '20250423',
                            '20250423_SR', 
                            '20250424',
                            '20250424_MC',
                            '20250424_SR',            
                            '20250425',
                            '20250425_SR',
                            '20250428_NV',
                            '20250428_MC',
                            '20250428_SR',  
                            '20250429_NV',
                            '20250429_MC',
                            '20250429_SR',  
                            '20250430_NV',
                            '20250430_MC',
                            '20250430_SR',  
            
                          ] # to deal with the sessions that MC and SR were in the same session
        session_start_times = [ 
                                0.00,
                                340,
                                340,
                                72.0,
                                60.1,
                                60.1,
                                82.2,
                                82.2,
                                35.8,
                                0.00,
                                29.2,
                                35.8,
                                62.5,
                                71.5,
                                54.4,
            
                                0.00,
                                0.00,
                                0.00,
            
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
            
                                0.00,
                                0.00,
                                73.5,
                                0.00,
                                76.1,
                                81.5,
                                0.00,
            
                                363,
                                # 0.00,
                                79.0,
                                162.6,
                                231.9,
                                109,
                                0.00,
            
                                0.00,
                                0.00, 
                                0.00,
                                0.00,
                                0.00,          
                                0.00,
                                93.0,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00, 
                                0.00,
                                274.4,
                                0.00,
            
                              ] # in second
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = [ 'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0',# 'Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             
                              ]
        animal1_fixedorders = ['dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson',# 'dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson',
                              ]
        recordedanimals = animal1_fixedorders 
        animal2_fixedorders = ['ginger','ginger','ginger','ginger','ginger','ginger','ginger','ginger','ginger',
                               'ginger','ginger','ginger','ginger','ginger','ginger','gingerNew','gingerNew','gingerNew',
                               'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala',
                               'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala',
                               'koala', 'koala', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', # 'kanga', 
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                              ]

        animal1_filenames = ["Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             'Dodson',# 'Dodson',
                             'Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson',
                             'Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson',
                             'Dodson','Dodson','Dodson','Dodson','Dodson',
                            ]
        animal2_filenames = ["Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger",
                             "Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger",
                             "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala",
                             "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala",
                             "Koala", "Koala", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga",
                             'Kanga', # 'Kanga', 
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                            ]
        
    elif do_OFC:
        # pick only five sessions for each conditions
        neural_record_conditions = [
                                     '20231101_Dodson_withGinger_MC',
                                     '20231107_Dodson_withGinger_MC',
                                     '20231122_Dodson_withGinger_MC',
                                     '20231129_Dodson_withGinger_MC',
                                     # '20231101_Dodson_withGinger_SR', # ginger didnot pull or only pulled once
                                     # '20231107_Dodson_withGinger_SR', # ginger didnot pull or only pulled once
                                     # '20231122_Dodson_withGinger_SR', # ginger didnot pull or only pulled once
                                     # '20231129_Dodson_withGinger_SR', # ginger didnot pull or only pulled once
                                   ]
        task_conditions = [
                            'MC',
                            'MC',
                            'MC',
                            'MC',
                            # 'SR',
                            # 'SR',
                            # 'SR',
                            # 'SR',
                          ]
        dates_list = [
                      "20231101_MC",
                      "20231107_MC",
                      "20231122_MC",
                      "20231129_MC",
                      # "20231101_SR",
                      # "20231107_SR",
                      # "20231122_SR",
                      # "20231129_SR",      
                     ]
        videodates_list = dates_list
        session_start_times = [ 
                                 0.00,   
                                 0.00,  
                                 0.00,  
                                 0.00, 
                                 # 0.00,   
                                 # 0.00,  
                                 # 0.00,  
                                 # 0.00, 
                              ] # in second
        kilosortvers = [ 
                         2, 
                         2, 
                         4, 
                         4,
                         # 2, 
                         # 2, 
                         # 4, 
                         # 4,
                       ]
    
        trig_channelnames = ['Dev1/ai0']*np.shape(dates_list)[0]
        animal1_fixedorders = ['dodson']*np.shape(dates_list)[0]
        recordedanimals = animal1_fixedorders
        animal2_fixedorders = ['ginger']*np.shape(dates_list)[0]

        animal1_filenames = ["Dodson"]*np.shape(dates_list)[0]
        animal2_filenames = ["Ginger"]*np.shape(dates_list)[0]

    
# dannon kanga
if 0:
    if do_DLPFC:
        neural_record_conditions = [
                                     '20240508_Kanga_SR',
                                     '20240509_Kanga_MC',
                                     '20240513_Kanga_MC',
                                     '20240514_Kanga_SR',
                                     '20240523_Kanga_MC',
                                     '20240524_Kanga_SR',
                                     '20240606_Kanga_MC',
                                     '20240613_Kanga_MC_DannonAuto',
                                     '20240614_Kanga_MC_DannonAuto',
                                     '20240617_Kanga_MC_DannonAuto',
                                     '20240618_Kanga_MC_KangaAuto',
                                     '20240619_Kanga_MC_KangaAuto',
                                     '20240620_Kanga_MC_KangaAuto',
                                     '20240621_1_Kanga_NoVis',
                                     '20240624_Kanga_NoVis',
                                     '20240626_Kanga_NoVis',
            
                                     '20240808_Kanga_MC_withGinger',
                                     '20240809_Kanga_MC_withGinger',
                                     '20240812_Kanga_MC_withGinger',
                                     '20240813_Kanga_MC_withKoala',
                                     '20240814_Kanga_MC_withKoala',
                                     '20240815_Kanga_MC_withKoala',
                                     '20240819_Kanga_MC_withVermelho',
                                     '20240821_Kanga_MC_withVermelho',
                                     '20240822_Kanga_MC_withVermelho',
            
                                     '20250415_Kanga_MC_withDodson',
                                     '20250416_Kanga_SR_withDodson',
                                     '20250417_Kanga_MC_withDodson',
                                     '20250418_Kanga_SR_withDodson',
                                     '20250421_Kanga_SR_withDodson',
                                     '20250422_Kanga_MC_withDodson',
                                     '20250422_Kanga_SR_withDodson',
            
                                    '20250423_Kanga_MC_withDodson',
                                    '20250423_Kanga_SR_withDodson', 
                                    '20250424_Kanga_NV_withDodson',
                                    '20250424_Kanga_MC_withDodson',
                                    '20250424_Kanga_SR_withDodson',            
                                    '20250425_Kanga_NV_withDodson',
                                    '20250425_Kanga_SR_withDodson',
                                    '20250428_Kanga_NV_withDodson',
                                    '20250428_Kanga_MC_withDodson',
                                    '20250428_Kanga_SR_withDodson',  
                                    '20250429_Kanga_NV_withDodson',
                                    '20250429_Kanga_MC_withDodson',
                                    '20250429_Kanga_SR_withDodson',  
                                    '20250430_Kanga_NV_withDodson',
                                    '20250430_Kanga_MC_withDodson',
                                    '20250430_Kanga_SR_withDodson',  
                                   ]
        dates_list = [
                      "20240508",
                      "20240509",
                      "20240513",
                      "20240514",
                      "20240523",
                      "20240524",
                      "20240606",
                      "20240613",
                      "20240614",
                      "20240617",
                      "20240618",
                      "20240619",
                      "20240620",
                      "20240621_1",
                      "20240624",
                      "20240626",
            
                      "20240808",
                      "20240809",
                      "20240812",
                      "20240813",
                      "20240814",
                      "20240815",
                      "20240819",
                      "20240821",
                      "20240822",
            
                      "20250415",
                      "20250416",
                      "20250417",
                      "20250418",
                      "20250421",
                      "20250422",
                      "20250422_SR",
            
                        '20250423',
                        '20250423_SR', 
                        '20250424',
                        '20250424_MC',
                        '20250424_SR',            
                        '20250425',
                        '20250425_SR',
                        '20250428_NV',
                        '20250428_MC',
                        '20250428_SR',  
                        '20250429_NV',
                        '20250429_MC',
                        '20250429_SR',  
                        '20250430_NV',
                        '20250430_MC',
                        '20250430_SR',  
                     ]
        videodates_list = dates_list
        task_conditions = [
                             'SR',
                             'MC',
                             'MC',
                             'SR',
                             'MC',
                             'SR',
                             'MC',
                             'MC_DannonAuto',
                             'MC_DannonAuto',
                             'MC_DannonAuto',
                             'MC_KangaAuto',
                             'MC_KangaAuto',
                             'MC_KangaAuto',
                             'NV',
                             'NV',
                             'NV',   
                            
                             'MC_withGinger',
                             'MC_withGinger',
                             'MC_withGinger',
                             'MC_withKoala',
                             'MC_withKoala',
                             'MC_withKoala',
                             'MC_withVermelho',
                             'MC_withVermelho',
                             'MC_withVermelho',
            
                             'MC_withDodson',
                             'SR_withDodson',
                             'MC_withDodson',
                             'SR_withDodson',
                             'SR_withDodson',
                             'MC_withDodson',
                             'SR_withDodson',
            
                             'MC_withDodson',
                            'SR_withDodson', 
                            'NV_withDodson',
                            'MC_withDodson',
                            'SR_withDodson',            
                            'NV_withDodson',
                            'SR_withDodson',
                            'NV_withDodson',
                            'MC_withDodson',
                            'SR_withDodson',  
                            'NV_withDodson',
                            'MC_withDodson',
                            'SR_withDodson',  
                            'NV_withDodson',
                            'MC_withDodson',
                            'SR_withDodson',  
                          ]
        session_start_times = [ 
                                 0.00,
                                 36.0,
                                 69.5,
                                 0.00,
                                 62.0,
                                 0.00,
                                 89.0,
                                 0.00,
                                 0.00,
                                 0.00,
                                 165.8,
                                 96.0, 
                                 0.00,
                                 0.00,
                                 0.00,
                                 48.0,
                                
                                 59.2,
                                 49.5,
                                 40.0,
                                 50.0,
                                 0.00,
                                 69.8,
                                 85.0,
                                 212.9,
                                 68.5,
            
                                 363,
                                 0.00,
                                 79.0,
                                 162.6,
                                 231.9,
                                 109,
                                 0.00,
            
                                0.00,
                                0.00, 
                                0.00,
                                0.00,
                                0.00,          
                                0.00,
                                93.0,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00, 
                                0.00,
                                274.4,
                                0.00,
                              ] # in second
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = ['Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai0','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                             'Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              ]
        
        animal1_fixedorders = ['dannon','dannon','dannon','dannon','dannon','dannon','dannon','dannon',
                               'dannon','dannon','dannon','dannon','dannon','dannon','dannon','dannon',
                               'ginger','ginger','ginger','koala','koala','koala','vermelho','vermelho',
                               'vermelho','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                              ]
        animal2_fixedorders = ['kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                              ]
        recordedanimals = animal2_fixedorders

        animal1_filenames = ["Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon",
                             "Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon",
                             "Ginger","Ginger","Ginger", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga",
                              "Kanga","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             
                            ]
        animal2_filenames = ["Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Koala","Koala","Koala","Vermelho","Vermelho",
                             "Vermelho","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                            ]
        
    elif do_OFC:
        # pick only five sessions for each conditions
        neural_record_conditions = [
                                     
                                   ]
        dates_list = [
                      
                     ]
        videodates_list = dates_list
        task_conditions = [
                           
                          ]
        session_start_times = [ 
                                
                              ] # in second
        kilosortvers = [ 

                       ]
    
        animal1_fixedorders = ['dannon']*np.shape(dates_list)[0]
        animal2_fixedorders = ['kanga']*np.shape(dates_list)[0]
        recordedanimals = animal2_fixedorders
        
        animal1_filenames = ["Dannon"]*np.shape(dates_list)[0]
        animal2_filenames = ["Kanga"]*np.shape(dates_list)[0]
    

    
# a test case
if 1: # kanga example
    neural_record_conditions = ['20240513_Kanga_MC']
    dates_list = ["20240513"]
    videodates_list = dates_list
    task_conditions = ['MC']
    session_start_times = [69.5] # in second
    kilosortvers = [4]
    trig_channelnames = ['Dev1/ai0']
    animal1_fixedorders = ['dannon']
    animal2_fixedorders = ['kanga']
    recordedanimals = animal2_fixedorders
    animal1_filenames = ["Dannon"]
    animal2_filenames = ["Kanga"]
    
if 0: # kanga example
    neural_record_conditions = ['20250415_Kanga_MC_withDodson']
    dates_list = ["20250415"]
    videodates_list = dates_list
    task_conditions = ['MC_withDodson']
    session_start_times = [363] # in second
    kilosortvers = [4]
    trig_channelnames = ['Dev1/ai9']
    animal1_fixedorders = ['dodson']
    animal2_fixedorders = ['kanga']
    recordedanimals = animal2_fixedorders
    animal1_filenames = ["Dodson"]
    animal2_filenames = ["Kanga"]

if 0: # dodson example 
    neural_record_conditions = ['20250415_Dodson_MC_withKanga']
    dates_list = ["20250415"]
    videodates_list = dates_list
    task_conditions = ['MC_withKanga']
    session_start_times = [363] # in second
    kilosortvers = [4]
    trig_channelnames = ['Dev1/ai0']
    animal1_fixedorders = ['dodson']
    recordedanimals = animal1_fixedorders
    animal2_fixedorders = ['kanga']
    animal1_filenames = ["Dodson"]
    animal2_filenames = ["Kanga"]
    
if 0: # dodson example 
    neural_record_conditions = ['20250430_Dodson_MC_withKanga']
    dates_list = ["20250430_MC"]
    videodates_list = dates_list
    task_conditions = ['MC_withKanga']
    session_start_times = [274.4] # in second
    kilosortvers = [4]
    trig_channelnames = ['Dev1/ai0']
    animal1_fixedorders = ['dodson']
    recordedanimals = animal1_fixedorders
    animal2_fixedorders = ['kanga']
    animal1_filenames = ["Dodson"]
    animal2_filenames = ["Kanga"]
    
ndates = np.shape(dates_list)[0]

session_start_frames = session_start_times * fps # fps is 30Hz

totalsess_time = 600

# video tracking results info
animalnames_videotrack = ['dodson','scorch'] # does not really mean dodson and scorch, instead, indicate animal1 and animal2
bodypartnames_videotrack = ['rightTuft','whiteBlaze','leftTuft','rightEye','leftEye','mouth']


# which camera to analyzed
cameraID = 'camera-2'
cameraID_short = 'cam2'

considerlevertube = 1
considertubeonly = 0

# location of levers and tubes for camera 2
# # camera 1
# lever_locs_camI = {'dodson':np.array([645,600]),'scorch':np.array([425,435])}
# tube_locs_camI  = {'dodson':np.array([1350,630]),'scorch':np.array([555,345])}
# # camera 2
# # location of the estimiated middle of the box
lever_locs_camI = {'dodson':np.array([1325,615]),'scorch':np.array([560,615])}
# # location of the estimated lever
# lever_locs_camI = {'dodson':np.array([1335,715]),'scorch':np.array([550,715])}
tube_locs_camI  = {'dodson':np.array([1550,515]),'scorch':np.array([350,515])}
# # old
# # lever_locs_camI = {'dodson':np.array([1335,715]),'scorch':np.array([550,715])}
# # tube_locs_camI  = {'dodson':np.array([1650,490]),'scorch':np.array([250,490])}
# # camera 3
# lever_locs_camI = {'dodson':np.array([1580,440]),'scorch':np.array([1296,540])}
# tube_locs_camI  = {'dodson':np.array([1470,375]),'scorch':np.array([805,475])}


if np.shape(session_start_times)[0] != np.shape(dates_list)[0]:
    exit()

    
# define bhv events summarizing variables     


# where to save the summarizing data
data_saved_folder = '/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/3d_recontruction_analysis_self_and_coop_task_data_saved/'

# neural data folder
neural_data_folder = '/gpfs/radev/pi/nandy/jadi_gibbs_data/Marmoset_neural_recording/'

    

In [ ]:
# basic behavior analysis (define time stamps for each bhv events, etc)

try:
    if redo_anystep:
        dummy
    
    dump
    
    #
    print('loading all data')
    
    # load saved data
    data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
    
    # with open(data_saved_subfolder+'/owgaze1_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
    #     owgaze1_num_all_dates = pickle.load(f)
    
        
    print('all data from all dates are loaded')

except:

    print('analyze all dates')

    for idate in np.arange(0,ndates,1):
    
        date_tgt = dates_list[idate]
        videodate_tgt = videodates_list[idate]
        
        neural_record_condition = neural_record_conditions[idate]
        
        session_start_time = session_start_times[idate]
        
        kilosortver = kilosortvers[idate]

        trig_channelname = trig_channelnames[idate]
        
        animal1_filename = animal1_filenames[idate]
        animal2_filename = animal2_filenames[idate]
        
        animal1_fixedorder = [animal1_fixedorders[idate]]
        animal2_fixedorder = [animal2_fixedorders[idate]]
        
        recordedanimal = recordedanimals[idate]

        # folder and file path
        camera12_analyzed_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/test_video_cooperative_task_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera12/"
        camera23_analyzed_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/test_video_cooperative_task_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera23/"
        
        # 
        try: 
            singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_167500"
            bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_80000"
                bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_cameraSep1shuffle1_150000"
                bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"                
            # get the bodypart data from files
            bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,videodate_tgt)
            video_file_original = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"
        except:
            singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_167500"
            bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_80000"
                bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_cameraSep1shuffle1_150000"
                bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            
            # get the bodypart data from files
            bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,videodate_tgt)
            video_file_original = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"        
        
        
        # load behavioral results
        try:
            bhv_data_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/marmoset_tracking_bhv_data_from_task_code/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
            trial_record_json = glob.glob(bhv_data_path +date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_TrialRecord_" + "*.json")
            bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_bhv_data_" + "*.json")
            session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_session_info_" + "*.json")
            ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_ni_data_" + "*.json")
            lever_reading_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_lever_reading_" + "*.json")
            #
            trial_record = pd.read_json(trial_record_json[0])
            bhv_data = pd.read_json(bhv_data_json[0])
            session_info = pd.read_json(session_info_json[0])
            lever_reading = pd.read_json(lever_reading_json[0])
            # 
            with open(ni_data_json[0]) as f:
                for line in f:
                    ni_data=json.loads(line)   
        except:
            bhv_data_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/marmoset_tracking_bhv_data_from_task_code/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
            trial_record_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_TrialRecord_" + "*.json")
            bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_bhv_data_" + "*.json")
            session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_session_info_" + "*.json")
            ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_ni_data_" + "*.json")
            lever_reading_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_lever_reading_" + "*.json")
            #
            trial_record = pd.read_json(trial_record_json[0])
            bhv_data = pd.read_json(bhv_data_json[0])
            session_info = pd.read_json(session_info_json[0])
            lever_reading = pd.read_json(lever_reading_json[0])
            #
            with open(ni_data_json[0]) as f:
                for line in f:
                    ni_data=json.loads(line)

        # get animal info from the session information
        animal1 = session_info['lever1_animal'][0].lower()
        animal2 = session_info['lever2_animal'][0].lower()

        
        # get task type and cooperation threshold
        try:
            coop_thres = session_info["pulltime_thres"][0]
            tasktype = session_info["task_type"][0]
        except:
            coop_thres = 0
            tasktype = 1
 

        # successful trial or not
        succtrial_ornot = np.array((trial_record['rewarded']>0).astype(int))
        succpull1_ornot = np.array((np.isin(bhv_data[bhv_data['behavior_events']==1]['trial_number'],trial_record[trial_record['rewarded']>0]['trial_number'])).astype(int))
        succpull2_ornot = np.array((np.isin(bhv_data[bhv_data['behavior_events']==2]['trial_number'],trial_record[trial_record['rewarded']>0]['trial_number'])).astype(int))
        succpulls_ornot = [succpull1_ornot,succpull2_ornot]
            
        # clean up the trial_record
        warnings.filterwarnings('ignore')
        trial_record_clean = pd.DataFrame(columns=trial_record.columns)
        # for itrial in np.arange(0,np.max(trial_record['trial_number']),1):
        for itrial in trial_record['trial_number']:
            # trial_record_clean.loc[itrial] = trial_record[trial_record['trial_number']==itrial+1].iloc[[0]]
            trial_record_clean = trial_record_clean.append(trial_record[trial_record['trial_number']==itrial].iloc[[0]])
        trial_record_clean = trial_record_clean.reset_index(drop = True)

        # change bhv_data time to the absolute time
        time_points_new = pd.DataFrame(np.zeros(np.shape(bhv_data)[0]),columns=["time_points_new"])
        # for itrial in np.arange(0,np.max(trial_record_clean['trial_number']),1):
        for itrial in np.arange(0,np.shape(trial_record_clean)[0],1):
            # ind = bhv_data["trial_number"]==itrial+1
            ind = bhv_data["trial_number"]==trial_record_clean['trial_number'][itrial]
            new_time_itrial = bhv_data[ind]["time_points"] + trial_record_clean["trial_starttime"].iloc[itrial]
            time_points_new["time_points_new"][ind] = new_time_itrial
        bhv_data["time_points"] = time_points_new["time_points_new"]
        bhv_data = bhv_data[bhv_data["time_points"] != 0]


        
        # load behavioral event results
        try:
            # dummy
            print('load social gaze with '+cameraID+' only of '+date_tgt)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_look_ornot.pkl', 'rb') as f:
                output_look_ornot = pickle.load(f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allvectors.pkl', 'rb') as f:
                output_allvectors = pickle.load(f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allangles.pkl', 'rb') as f:
                output_allangles = pickle.load(f)  
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_key_locations.pkl', 'rb') as f:
                output_key_locations = pickle.load(f)
        except:   
            print('analyze social gaze with '+cameraID+' only of '+date_tgt)
            # get social gaze information 
            output_look_ornot, output_allvectors, output_allangles = find_socialgaze_timepoint_singlecam_wholebody(bodyparts_locs_camI,lever_locs_camI,tube_locs_camI,
                                                                                                                   considerlevertube,considertubeonly,sqr_thres_tubelever,
                                                                                                                   sqr_thres_face,sqr_thres_body)
            output_key_locations = find_socialgaze_timepoint_singlecam_wholebody_2(bodyparts_locs_camI,lever_locs_camI,tube_locs_camI,considerlevertube)
            
            # save data
            current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody/'+animal1_fixedorder[0]+animal2_fixedorder[0]
            add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
            if not os.path.exists(add_date_dir):
                os.makedirs(add_date_dir)
            #
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_look_ornot.pkl', 'wb') as f:
                pickle.dump(output_look_ornot, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allvectors.pkl', 'wb') as f:
                pickle.dump(output_allvectors, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allangles.pkl', 'wb') as f:
                pickle.dump(output_allangles, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_key_locations.pkl', 'wb') as f:
                pickle.dump(output_key_locations, f)
                

        look_at_other_or_not_merge = output_look_ornot['look_at_other_or_not_merge']
        look_at_tube_or_not_merge = output_look_ornot['look_at_tube_or_not_merge']
        look_at_lever_or_not_merge = output_look_ornot['look_at_lever_or_not_merge']
        look_at_otherlever_or_not_merge = output_look_ornot['look_at_otherlever_or_not_merge']
        look_at_otherface_or_not_merge = output_look_ornot['look_at_otherface_or_not_merge']
        
        # change the unit to second and align to the start of the session
        session_start_time = session_start_times[idate]
        look_at_other_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_other_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_lever_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_lever_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_tube_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_tube_or_not_merge['dodson'])[0],1)/fps - session_start_time 
        look_at_otherlever_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_otherlever_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_otherface_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_otherface_or_not_merge['dodson'])[0],1)/fps - session_start_time

        
        # find time point of behavioral events
        output_time_points_socialgaze ,output_time_points_levertube = bhv_events_timepoint_singlecam(bhv_data,look_at_other_or_not_merge,look_at_lever_or_not_merge,look_at_tube_or_not_merge)
        time_point_pull1 = output_time_points_socialgaze['time_point_pull1']
        time_point_pull2 = output_time_points_socialgaze['time_point_pull2']
        oneway_gaze1 = output_time_points_socialgaze['oneway_gaze1']
        oneway_gaze2 = output_time_points_socialgaze['oneway_gaze2']
        mutual_gaze1 = output_time_points_socialgaze['mutual_gaze1']
        mutual_gaze2 = output_time_points_socialgaze['mutual_gaze2']
        # 
        # mostly just for the sessions in which MC and SR are in the same session 
        firstpulltime = np.nanmin([np.nanmin(time_point_pull1),np.nanmin(time_point_pull2)])
        oneway_gaze1 = oneway_gaze1[oneway_gaze1>(firstpulltime-15)] # 15s before the first pull (animal1 or 2) count as the active period
        oneway_gaze2 = oneway_gaze2[oneway_gaze2>(firstpulltime-15)]
        mutual_gaze1 = mutual_gaze1[mutual_gaze1>(firstpulltime-15)]
        mutual_gaze2 = mutual_gaze2[mutual_gaze2>(firstpulltime-15)]  
        #    
        # newly added condition: only consider gaze during the active pulling time (15s after the last pull)    
        lastpulltime = np.nanmax([np.nanmax(time_point_pull1),np.nanmax(time_point_pull2)])
        oneway_gaze1 = oneway_gaze1[oneway_gaze1<(lastpulltime+15)]    
        oneway_gaze2 = oneway_gaze2[oneway_gaze2<(lastpulltime+15)]
        mutual_gaze1 = mutual_gaze1[mutual_gaze1<(lastpulltime+15)]
        mutual_gaze2 = mutual_gaze2[mutual_gaze2<(lastpulltime+15)] 
            
        # define successful pulls and failed pulls
        if 0: # old definition; not in use
            trialnum_succ = np.array(trial_record_clean['trial_number'][trial_record_clean['rewarded']>0])
            bhv_data_succ = bhv_data[np.isin(bhv_data['trial_number'],trialnum_succ)]
            #
            time_point_pull1_succ = bhv_data_succ["time_points"][bhv_data_succ["behavior_events"]==1]
            time_point_pull2_succ = bhv_data_succ["time_points"][bhv_data_succ["behavior_events"]==2]
            time_point_pull1_succ = np.round(time_point_pull1_succ,1)
            time_point_pull2_succ = np.round(time_point_pull2_succ,1)
            #
            trialnum_fail = np.array(trial_record_clean['trial_number'][trial_record_clean['rewarded']==0])
            bhv_data_fail = bhv_data[np.isin(bhv_data['trial_number'],trialnum_fail)]
            #
            time_point_pull1_fail = bhv_data_fail["time_points"][bhv_data_fail["behavior_events"]==1]
            time_point_pull2_fail = bhv_data_fail["time_points"][bhv_data_fail["behavior_events"]==2]
            time_point_pull1_fail = np.round(time_point_pull1_fail,1)
            time_point_pull2_fail = np.round(time_point_pull2_fail,1)
        else:
            # a new definition of successful and failed pulls
            # separate successful and failed pulls
            # step 1 all pull and juice
            time_point_pull1 = bhv_data["time_points"][bhv_data["behavior_events"]==1]
            time_point_pull2 = bhv_data["time_points"][bhv_data["behavior_events"]==2]
            time_point_juice1 = bhv_data["time_points"][bhv_data["behavior_events"]==3]
            time_point_juice2 = bhv_data["time_points"][bhv_data["behavior_events"]==4]
            # step 2:
            # pull 1
            # Find the last pull before each juice
            successful_pull1 = [time_point_pull1[time_point_pull1 < juice].max() for juice in time_point_juice1]
            # Convert to Pandas Series
            successful_pull1 = pd.Series(successful_pull1, index=time_point_juice1.index)
            # Find failed pulls (pulls that are not successful)
            failed_pull1 = time_point_pull1[~time_point_pull1.isin(successful_pull1)]
            # pull 2
            # Find the last pull before each juice
            successful_pull2 = [time_point_pull2[time_point_pull2 < juice].max() for juice in time_point_juice2]
            # Convert to Pandas Series
            successful_pull2 = pd.Series(successful_pull2, index=time_point_juice2.index)
            # Find failed pulls (pulls that are not successful)
            failed_pull2 = time_point_pull2[~time_point_pull2.isin(successful_pull2)]
            #
            # step 3:
            time_point_pull1_succ = np.round(successful_pull1,1)
            time_point_pull2_succ = np.round(successful_pull2,1)
            time_point_pull1_fail = np.round(failed_pull1,1)
            time_point_pull2_fail = np.round(failed_pull2,1)
        # 
        time_point_pulls_succfail = { "pull1_succ":time_point_pull1_succ,
                                      "pull2_succ":time_point_pull2_succ,
                                      "pull1_fail":time_point_pull1_fail,
                                      "pull2_fail":time_point_pull2_fail,
                                    }


    
            
        # new total session time (instead of 600s) - total time of the video recording
        totalsess_time = np.floor(np.shape(output_look_ornot['look_at_lever_or_not_merge']['dodson'])[0]/30) 
                

    # save data
    if 0:
        
        data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
        if not os.path.exists(data_saved_subfolder):
            os.makedirs(data_saved_subfolder)
                
        # with open(data_saved_subfolder+'/DBN_input_data_alltypes_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
        #     pickle.dump(DBN_input_data_alltypes, f)

    

In [ ]:
# ===== Optic flow demo video around a single pull (copy/paste full cell) =====
if 0:
    import os
    import numpy as np
    import cv2

    import matplotlib.animation as animation  # (kept from your original cell; not required for the demo video)
    import matplotlib.pyplot as plt

    # -----------------------------
    # Settings / paths (from your original snippet)
    # -----------------------------
    # video file path for saving / locating

    fps = 30

    # NOTE: your snippet had vidcap = cv2.VideoCapture(video_file_original)
    # For this demo, we will use `video_file` above.
    # If you truly want the "original" file, set video_file_original and replace video_in below.
    video_in = video_file_original  # or video_file_original

    # -----------------------------
    # (Optional) Your original writer block (not used for the demo video)
    # -----------------------------
    FFMpegWriter = animation.writers['ffmpeg']
    metadata = dict(title='Animal tracking demo', artist='Matplotlib', comment='')
    writer = FFMpegWriter(fps=fps, metadata=metadata)

    # -----------------------------
    # Your original variables (kept; not required for optic flow demo)
    # -----------------------------
    # animal_names_unique = pd.unique(pd.DataFrame(bodyparts_locs_camN.keys()).iloc[:,0])
    # body_parts_unique = pd.unique(pd.DataFrame(bodyparts_locs_camN.keys()).iloc[:,1])
    animal_names_unique = animalnames_videotrack
    body_parts_unique = bodypartnames_videotrack

    nanimals = np.shape(animal_names_unique)[0]
    nbodyparts = np.shape(body_parts_unique)[0]

    # -----------------------------
    # Align window: choose pull and animal half-screen crop (from your snippet)
    # -----------------------------
    pullid_tgt = 46 # animal 1: good example 40; animal 2: good example 46
    animalID = 2  # 1: animal on the right; 2: animal on the left

    nframes = int(fps * 8)  # -4 to +4s

    xpoint_crop = 950  # for animal 1, crop the right part of the screen and vice versa

    # Compute iframe_min / iframe_max
    if animalID == 1:
        iframe_min = int(np.round(session_start_time * fps)) + int((np.array(time_point_pull1)[pullid_tgt] - 4) * fps)
    elif animalID == 2:
        iframe_min = int(np.round(session_start_time * fps)) + int((np.array(time_point_pull2)[pullid_tgt] - 4) * fps)
    else:
        raise ValueError("animalID must be 1 (right animal) or 2 (left animal)")

    iframe_max = int(nframes + iframe_min)

    print(f"Video in: {video_in}")
    print(f"iframe_min={iframe_min}, iframe_max={iframe_max}, nframes={iframe_max-iframe_min}")

    # -----------------------------
    # Optic flow demo video output
    # -----------------------------
    # Write to a convenient local path (change if you prefer)
    video_out = f"/opticflow_demo_{date_tgt}_animal{animalID}_pull{pullid_tgt}.mp4"

    # Visualization scaling: tune if flow looks too dark/bright
    mag_clip = 8.0  # lower -> brighter; higher -> dimmer
    show_text = True

    # -----------------------------
    # Open video and define crop
    # -----------------------------
    cap = cv2.VideoCapture(video_in)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_in}")

    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Define half-screen crop boundaries
    if animalID == 1:
        x0, x1 = int(xpoint_crop), W   # right half
    else:
        x0, x1 = 0, int(xpoint_crop)   # left half

    crop_w = x1 - x0
    crop_h = H

    # Output video will be side-by-side: [original crop | flow visualization]
    out_w = crop_w * 2
    out_h = crop_h

    # Video writer
    # -----------------------------
    # Robust VideoWriter setup (FORCES container via extension)
    # -----------------------------
    out_base = f"opticflow_demo_{date_tgt}_animal{animalID}_pull{pullid_tgt}"  # no extension here!

    fourcc_candidates = [
        ("XVID", ".avi"),
        ("MJPG", ".avi"),
        ("mp4v", ".mp4"),
    ]

    vw = None
    video_out = None

    for fourcc_str, ext in fourcc_candidates:
        fourcc = cv2.VideoWriter_fourcc(*fourcc_str)
        video_out_try = out_base + ext  # FORCE container by extension

        vw_try = cv2.VideoWriter(video_out_try, fourcc, fps, (out_w, out_h))
        if vw_try.isOpened():
            vw = vw_try
            video_out = video_out_try
            print(f"VideoWriter opened with codec={fourcc_str}, output={video_out}")
            break
        else:
            vw_try.release()
            print(f"Failed to open VideoWriter with codec={fourcc_str}, output={video_out_try}")

    if vw is None or (not vw.isOpened()):
        cap.release()
        raise RuntimeError("Could not open VideoWriter with XVID/MJPG/MP4V. "
                           "Your OpenCV/FFMPEG build likely cannot encode video on this node.")





    # Seek to iframe_min
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(iframe_min))

    ok, frame0 = cap.read()
    if not ok:
        cap.release()
        vw.release()
        raise RuntimeError(f"Failed to read at iframe_min={iframe_min}")

    crop0 = frame0[:, x0:x1]
    prev_gray = cv2.cvtColor(crop0, cv2.COLOR_BGR2GRAY)

    def flow_to_hsv_bgr(flow, mag_clip=8.0):
        """Convert optical flow to BGR image: hue=direction, value=magnitude."""
        fx, fy = flow[..., 0], flow[..., 1]
        mag, ang = cv2.cartToPolar(fx, fy, angleInDegrees=False)

        hsv = np.zeros((flow.shape[0], flow.shape[1], 3), dtype=np.uint8)
        hsv[..., 0] = (ang * 180 / np.pi / 2).astype(np.uint8)  # OpenCV hue in [0,180]
        hsv[..., 1] = 255

        mag_norm = np.clip(mag / mag_clip, 0, 1)
        hsv[..., 2] = (mag_norm * 255).astype(np.uint8)

        bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
        return bgr, mag

    def flow_to_gray(flow, mag_clip=8.0):
        """
        Convert optical flow to grayscale motion-energy image.
        White = high motion, black = low motion.
        """
        fx, fy = flow[..., 0], flow[..., 1]
        mag = np.sqrt(fx**2 + fy**2)

        # Normalize and clip
        mag_norm = np.clip(mag / mag_clip, 0, 1)

        # Convert to 8-bit grayscale
        gray = (mag_norm * 255).astype(np.uint8)

        # Convert to 3-channel BGR so we can concatenate with RGB frames
        gray_bgr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        return gray_bgr, mag


    # -----------------------------
    # Main loop: compute optic flow and write demo frames
    # -----------------------------
    nwin = int(iframe_max - iframe_min)
    # first frame has no flow yet; we’ll start writing from frame1 (i=1)
    for i in range(1, nwin):
        ok, frame = cap.read()
        if not ok:
            print(f"Stopped early at i={i} (couldn't read frame).")
            break

        crop = frame[:, x0:x1]
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

        # Farneback optical flow
        flow = cv2.calcOpticalFlowFarneback(
            prev_gray, gray, None,
            pyr_scale=0.5, levels=3, winsize=15,
            iterations=3, poly_n=5, poly_sigma=1.2, flags=0
        )

        # flow_vis, mag = flow_to_hsv_bgr(flow, mag_clip=mag_clip)
        flow_vis, mag = flow_to_gray(flow, mag_clip=mag_clip)

        crop_disp = crop.copy()
        flow_disp = flow_vis.copy()

        # time relative to pull (window is -4 to +4)
        t_rel = -4.0 + (i / fps)

        if show_text:
            mean_mag = float(np.mean(mag))
            cv2.putText(crop_disp, f"t = {t_rel:+.3f}s", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2, cv2.LINE_AA)
            cv2.putText(flow_disp, f"mean|flow| = {mean_mag:.3f}", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2, cv2.LINE_AA)

        side_by_side = np.concatenate([crop_disp, flow_disp], axis=1)
        vw.write(side_by_side)

        prev_gray = gray

    cap.release()
    vw.release()

    print("Saved optic flow demo video to:", video_out)

    # (Optional) quick display of the first frame crop to confirm ROI
    # Uncomment if you want a sanity check figure in the notebook:
    # cap = cv2.VideoCapture(video_in)
    # cap.set(cv2.CAP_PROP_POS_FRAMES, int(iframe_min))
    # ok, f = cap.read()
    # cap.release()
    # if ok:
    #     plt.figure(figsize=(8,4))
    #     plt.imshow(cv2.cvtColor(f[:, x0:x1], cv2.COLOR_BGR2RGB))
    #     plt.title("Crop used for optic flow demo")
    #     plt.axis("off")
    #     plt.show()


In [ ]:
# np.array(time_point_pull1)

In [ ]:
# ===== Face-anchored optic flow demo video + trace + VECTOR-PDF snapshots at t=-1,0,+1 =====
# Vector-PDF snapshots: raster images embedded, but ROI boxes + trace lines + text are vector-editable.

if 1:
    import numpy as np
    import cv2
    import matplotlib.pyplot as plt
    from matplotlib.backends.backend_pdf import PdfPages
    import matplotlib.patches as patches
    import subprocess

    # -----------------------------
    # USER SETTINGS
    # -----------------------------
    fps = 30
    pullid_tgt = 15
    animalID = 2  # 1: crop right half; 2: crop left half
    
    xpoint_crop = 950
    sqr_thres_face = 1.30

    fb_params = dict(
        pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0
    )

    mag_clip = 8.0  # for visual scaling of motion-energy

    # Trace panel parameters (only used for video, snapshots use matplotlib vector traces)
    trace_height = 300
    trace_width  = 560
    trace_margin = 25

    # Input video + tracking dict
    video_in = video_file_original
    bp = bodyparts_locs_camI

    if animalID == 1:
        ianimal_name = "dodson"
    elif animalID == 2:
        ianimal_name = "scorch"
    else:
        raise ValueError("animalID must be 1 or 2")

    # Pull-aligned window: -4 to +4 s
    if animalID == 1:
        iframe_min = int(np.round(session_start_time * fps)) + int((np.array(time_point_pull1)[pullid_tgt] - 4) * fps)
    else:
        iframe_min = int(np.round(session_start_time * fps)) + int((np.array(time_point_pull2)[pullid_tgt] - 4) * fps)
    iframe_max = int(iframe_min + fps * 8)

    print(f"Video in: {video_in}")
    print(f"Tracking animal: {ianimal_name}")
    print(f"iframe_min={iframe_min}, iframe_max={iframe_max}, nframes={iframe_max-iframe_min}")

    # -----------------------------
    # Snapshot settings (t=-1,0,+1)
    # loop uses t_rel = -4 + i/fps, i starts at 1
    # -----------------------------
    snapshot_times = [-1.0, 0.0, 1.0]
    snapshot_frame_idx = {t: int(round((t + 4.0) * fps)) for t in snapshot_times}  # i index
    print("Snapshot frame indices (loop i):", snapshot_frame_idx)

    snapshots = {}  # t -> dict with raw components for vector-PDF render

    # -----------------------------
    # Helper functions (preserved)
    # -----------------------------
    def safe_get_xy(arr, iframe):
        if arr is None:
            return np.array([np.nan, np.nan], dtype=float)
        try:
            xy = np.array(arr[iframe, :], dtype=float)
            if xy.shape[0] != 2:
                return np.array([np.nan, np.nan], dtype=float)
            return xy
        except Exception:
            return np.array([np.nan, np.nan], dtype=float)

    def clip_box_center(xc, yc, half, w, h):
        if np.any(np.isnan([xc, yc, half])) or half <= 1:
            return None
        x0 = int(np.floor(xc - half)); x1 = int(np.ceil(xc + half))
        y0 = int(np.floor(yc - half)); y1 = int(np.ceil(yc + half))
        x0 = max(0, min(w-1, x0)); x1 = max(1, min(w, x1))
        y0 = max(0, min(h-1, y0)); y1 = max(1, min(h, y1))
        if (x1-x0) < 2 or (y1-y0) < 2:
            return None
        return (x0, x1, y0, y1)

    def clip_box_xyxy(x0, x1, y0, y1, w, h):
        x0 = int(max(0, min(w-1, x0))); x1 = int(max(1, min(w, x1)))
        y0 = int(max(0, min(h-1, y0))); y1 = int(max(1, min(h, y1)))
        if (x1-x0) < 2 or (y1-y0) < 2:
            return None
        return (x0, x1, y0, y1)

    def mean_in_box(mag_map, box):
        if box is None:
            return np.nan
        x0,x1,y0,y1 = box
        patch = mag_map[y0:y1, x0:x1]
        if patch.size < 4:
            return np.nan
        return float(np.nanmean(patch))

    def draw_box_cv(img, box, color, thickness=2, label=None):
        if box is None:
            return
        x0,x1,y0,y1 = box
        cv2.rectangle(img, (x0,y0), (x1,y1), color, thickness)
        if label is not None:
            cv2.putText(img, label, (x0+5, max(20, y0+20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)

    def pad_to_height(img, target_h):
        h, w = img.shape[:2]
        if h == target_h:
            return img
        pad = np.zeros((target_h - h, w, 3), dtype=np.uint8)
        return np.vstack([img, pad])

    # -----------------------------
    # Open video + define crop
    # -----------------------------
    cap = cv2.VideoCapture(video_in)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_in}")

    W_full = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H_full = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if animalID == 1:
        X0, X1 = int(xpoint_crop), W_full
    else:
        X0, X1 = 0, int(xpoint_crop)

    crop_w = X1 - X0
    crop_h = H_full

    # Video output layout: [orig crop | motion gray | trace bitmap]
    out_w = crop_w + crop_w + trace_width
    out_h = max(crop_h, trace_height)

    # -----------------------------
    # VideoWriter (kept)
    # -----------------------------
    out_base = f"opticflow_face_LRU_traces_pullmark_{date_tgt}_animal{animalID}_pull{pullid_tgt}"
    fourcc_candidates = [("XVID", ".avi"), ("MJPG", ".avi"), ("mp4v", ".mp4")]

    vw = None
    video_out = None
    for fourcc_str, ext in fourcc_candidates:
        fourcc = cv2.VideoWriter_fourcc(*fourcc_str)
        video_out_try = out_base + ext
        vw_try = cv2.VideoWriter(video_out_try, fourcc, fps, (out_w, out_h))
        if vw_try.isOpened():
            vw = vw_try
            video_out = video_out_try
            print(f"VideoWriter opened with codec={fourcc_str}, output={video_out}")
            break
        else:
            vw_try.release()

    if vw is None or (not vw.isOpened()):
        cap.release()
        raise RuntimeError("Could not open VideoWriter with XVID/MJPG/MP4V on this node.")

    # -----------------------------
    # Seek + init
    # -----------------------------
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(iframe_min))
    ok, frame0 = cap.read()
    if not ok:
        cap.release()
        vw.release()
        raise RuntimeError(f"Failed to read at iframe_min={iframe_min}")

    crop0 = frame0[:, X0:X1]
    prev_gray = cv2.cvtColor(crop0, cv2.COLOR_BGR2GRAY)

    # trace storage (for the video + for snapshot trace)
    t_rel_list = []
    face_motion_list = []
    left_motion_list = []
    right_motion_list = []
    under_motion_list = []

    # -----------------------------
    # Tiny helper to make the bitmap trace panel for VIDEO ONLY (kept simple)
    # -----------------------------
    def make_trace_panel_bitmap(t_rel_arr, face_arr, left_arr, right_arr, under_arr, t_now,
                                H=300, W=560, margin=25):
        panel = np.zeros((H, W, 3), dtype=np.uint8)

        all_vals = np.concatenate([face_arr, left_arr, right_arr, under_arr])
        finite = np.isfinite(all_vals)
        vmax = np.nanpercentile(all_vals[finite], 99) if np.any(finite) else 1.0
        vmax = max(vmax, 1e-6)

        x0 = margin; x1 = W - margin
        y0 = margin; y1 = H - margin
        cv2.rectangle(panel, (x0, y0), (x1, y1), (200, 200, 200), 1)

        tmin, tmax = -4.0, 4.0
        def tx(t):
            return int(x0 + (t - tmin) / (tmax - tmin) * (x1 - x0))
        def ty(v):
            v = np.clip(v, 0, vmax)
            return int(y1 - (v / vmax) * (y1 - y0))

        def draw_trace(series, color):
            pts = []
            for t, v in zip(t_rel_arr, series):
                if np.isnan(v):
                    continue
                pts.append((tx(t), ty(v)))
            if len(pts) >= 2:
                cv2.polylines(panel, [np.array(pts, dtype=np.int32)], False, color, 2)

        # pull marker
        x_pull = tx(0.0)
        for yy in range(y0, y1, 8):
            cv2.line(panel, (x_pull, yy), (x_pull, min(yy + 4, y1)), (255, 255, 0), 2)

        draw_trace(left_arr,  (0, 0, 255))
        draw_trace(right_arr, (255, 0, 0))
        draw_trace(under_arr, (0, 255, 0))
        draw_trace(face_arr,  (255, 255, 255))

        # cursor
        xc = tx(t_now)
        cv2.line(panel, (xc, y0), (xc, y1), (0, 255, 255), 1)

        return panel

    # -----------------------------
    # Main loop
    # -----------------------------
    nwin = int(iframe_max - iframe_min)
    for i in range(1, nwin):
        ok, frame = cap.read()
        if not ok:
            print(f"Stopped early at i={i} (couldn't read frame).")
            break

        iframe = int(iframe_min + i)
        crop = frame[:, X0:X1]
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

        flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, **fb_params)

        # motion magnitude (normalized) for visualization + ROI averaging
        fx, fy = flow[..., 0], flow[..., 1]
        mag = np.sqrt(fx*fx + fy*fy)
        mag_norm = np.clip(mag / mag_clip, 0, 1)   # 0..1 float
        motion_gray_u8 = (mag_norm * 255).astype(np.uint8)
        flow_vis_bgr = cv2.cvtColor(motion_gray_u8, cv2.COLOR_GRAY2BGR)

        # --- Face ROI from DLC landmarks (FULL-FRAME coords) ---
        rightTuft  = safe_get_xy(bp.get((ianimal_name, 'rightTuft')), iframe)
        leftTuft   = safe_get_xy(bp.get((ianimal_name, 'leftTuft')), iframe)
        whiteBlaze = safe_get_xy(bp.get((ianimal_name, 'whiteBlaze')), iframe)
        rightEye   = safe_get_xy(bp.get((ianimal_name, 'rightEye')), iframe)
        leftEye    = safe_get_xy(bp.get((ianimal_name, 'leftEye')), iframe)
        mouth      = safe_get_xy(bp.get((ianimal_name, 'mouth')), iframe)

        face_mass = np.nanmean(np.vstack([rightTuft, whiteBlaze, leftTuft, leftEye, mouth, rightEye]), axis=0)

        dist7  = np.linalg.norm(rightEye - rightTuft) if np.all(np.isfinite(rightEye)) and np.all(np.isfinite(rightTuft)) else np.nan
        dist8  = np.linalg.norm(leftEye  - leftTuft ) if np.all(np.isfinite(leftEye )) and np.all(np.isfinite(leftTuft )) else np.nan
        dist9  = np.linalg.norm(whiteBlaze - rightTuft) if np.all(np.isfinite(whiteBlaze)) and np.all(np.isfinite(rightTuft)) else np.nan
        dist10 = np.linalg.norm(whiteBlaze - leftTuft ) if np.all(np.isfinite(whiteBlaze)) and np.all(np.isfinite(leftTuft )) else np.nan

        face_offset = np.nanmax([dist7, dist8, dist9, dist10]) * sqr_thres_face

        face_x = face_mass[0] - X0
        face_y = face_mass[1]
        face_box = clip_box_center(face_x, face_y, face_offset, crop_w, crop_h)

        roi_boxes = {"face": face_box, "left": None, "right": None, "under": None}
        face_mean = left_mean = right_mean = under_mean = np.nan

        if face_box is not None:
            face_mean = mean_in_box(mag_norm, face_box)

            fx0, fx1, fy0, fy1 = face_box
            fw = fx1 - fx0
            fh = fy1 - fy0

            left_box  = clip_box_xyxy(fx0 - fw, fx0, fy0, fy0 + fh, crop_w, crop_h)
            right_box = clip_box_xyxy(fx1, fx1 + fw, fy0, fy0 + fh, crop_w, crop_h)
            under_box = clip_box_xyxy(fx0, fx1, fy1, fy1 + fh, crop_w, crop_h)

            roi_boxes["left"]  = left_box
            roi_boxes["right"] = right_box
            roi_boxes["under"] = under_box

            left_mean  = mean_in_box(mag_norm, left_box)
            right_mean = mean_in_box(mag_norm, right_box)
            under_mean = mean_in_box(mag_norm, under_box)

        # time relative to pull
        t_rel = -4.0 + (i / fps)
        t_rel_list.append(t_rel)
        face_motion_list.append(face_mean)
        left_motion_list.append(left_mean)
        right_motion_list.append(right_mean)
        under_motion_list.append(under_mean)

        # ----- VIDEO FRAME (bitmap) -----
        crop_disp = crop.copy()
        flow_disp = flow_vis_bgr.copy()

        draw_box_cv(crop_disp, roi_boxes["face"], (255,255,255), 2, label="face")
        draw_box_cv(crop_disp, roi_boxes["left"], (0,0,255), 2, label="L")
        draw_box_cv(crop_disp, roi_boxes["right"], (255,0,0), 2, label="R")
        draw_box_cv(crop_disp, roi_boxes["under"], (0,255,0), 2, label="U")

        draw_box_cv(flow_disp, roi_boxes["face"], (255,255,255), 2)
        draw_box_cv(flow_disp, roi_boxes["left"], (0,0,255), 2)
        draw_box_cv(flow_disp, roi_boxes["right"], (255,0,0), 2)
        draw_box_cv(flow_disp, roi_boxes["under"], (0,255,0), 2)

        t_arr = np.array(t_rel_list, dtype=float)
        f_arr = np.array(face_motion_list, dtype=float)
        l_arr = np.array(left_motion_list, dtype=float)
        r_arr = np.array(right_motion_list, dtype=float)
        u_arr = np.array(under_motion_list, dtype=float)

        trace_bitmap = make_trace_panel_bitmap(t_arr, f_arr, l_arr, r_arr, u_arr, t_rel,
                                               H=trace_height, W=trace_width, margin=trace_margin)

        crop_p  = pad_to_height(crop_disp, out_h)
        flow_p  = pad_to_height(flow_disp, out_h)
        trace_p = pad_to_height(trace_bitmap, out_h)

        out_frame = np.concatenate([crop_p, flow_p, trace_p], axis=1)
        vw.write(out_frame)

        # ----- SNAPSHOT CAPTURE (store RAW components for vector-PDF render) -----
        for t_snap, idx in snapshot_frame_idx.items():
            if i == idx and t_snap not in snapshots:
                snapshots[t_snap] = dict(
                    crop_bgr=crop.copy(),
                    mag_norm=mag_norm.copy(),      # 2D float 0..1
                    roi_boxes={k: (None if v is None else tuple(v)) for k, v in roi_boxes.items()},
                    t_arr=t_arr.copy(),
                    face=f_arr.copy(),
                    left=l_arr.copy(),
                    right=r_arr.copy(),
                    under=u_arr.copy(),
                    t_now=t_rel
                )
                print(f"Captured VECTOR snapshot data at t={t_snap:+.1f}s (i={i})")

        prev_gray = gray

    cap.release()
    vw.release()
    print("Saved video to:", video_out)

    # -----------------------------
    # SAVE VECTOR-EDITABLE SNAPSHOTS AS PDF
    # (images are raster, overlays + traces + text are vector)
    # -----------------------------
    def add_roi_rect(ax, box, color, label=None, lw=2):
        if box is None:
            return
        x0, x1, y0, y1 = box
        rect = patches.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, linewidth=lw, edgecolor=color)
        ax.add_patch(rect)
        if label is not None:
            ax.text(x0 + 5, y0 + 20, label, color=color, fontsize=12, weight="bold")

    if len(snapshots) > 0:
        pdf_name = f"opticflow_vector_snapshots_{date_tgt}_animal{animalID}_pull{pullid_tgt}.pdf"

        with PdfPages(pdf_name) as pdf:
            for t_snap in sorted(snapshots.keys()):
                S = snapshots[t_snap]

                crop_rgb = cv2.cvtColor(S["crop_bgr"], cv2.COLOR_BGR2RGB)
                mag_u8 = (np.clip(S["mag_norm"], 0, 1) * 255).astype(np.uint8)

                fig = plt.figure(figsize=(14, 5.2))
                gs = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.0, 1.2], wspace=0.02)

                ax1 = fig.add_subplot(gs[0,0])
                ax2 = fig.add_subplot(gs[0,1])
                ax3 = fig.add_subplot(gs[0,2])

                # --- Panel 1: original crop (raster), ROI boxes vector ---
                ax1.imshow(crop_rgb)
                ax1.axis("off")
                add_roi_rect(ax1, S["roi_boxes"]["face"],  "white", "face")
                add_roi_rect(ax1, S["roi_boxes"]["left"],  "red",   "L")
                add_roi_rect(ax1, S["roi_boxes"]["right"], "blue",  "R")
                add_roi_rect(ax1, S["roi_boxes"]["under"], "lime",  "U")
                ax1.set_title(f"Crop (t={t_snap:+.1f}s)", fontsize=12)

                # --- Panel 2: motion magnitude (raster), ROI boxes vector ---
                ax2.imshow(mag_u8, cmap="gray", vmin=0, vmax=255)
                ax2.axis("off")
                add_roi_rect(ax2, S["roi_boxes"]["face"],  "white")
                add_roi_rect(ax2, S["roi_boxes"]["left"],  "red")
                add_roi_rect(ax2, S["roi_boxes"]["right"], "blue")
                add_roi_rect(ax2, S["roi_boxes"]["under"], "lime")
                ax2.set_title("Optic-flow magnitude", fontsize=12)

                # --- Panel 3: traces (fully vector) ---
                ax3.plot(S["t_arr"], S["left"],  color="red",  label="L", linewidth=2)
                ax3.plot(S["t_arr"], S["right"], color="blue", label="R", linewidth=2)
                ax3.plot(S["t_arr"], S["under"], color="green",label="U", linewidth=2)
                ax3.plot(S["t_arr"], S["face"],  color="black",label="face", linewidth=2.5)

                ax3.axvline(0, color="gold", linestyle="--", linewidth=2, label="pull")
                ax3.axvline(S["t_now"], color="cyan", linestyle="-", linewidth=1)

                ax3.set_xlim(-4, 4)
                ax3.set_xlabel("Time from pull (s)")
                ax3.set_ylabel("Mean motion (norm.)")
                ax3.legend(frameon=False, fontsize=10, loc="upper right")
                ax3.set_title("ROI mean motion traces", fontsize=12)

                fig.suptitle(f"Optic-flow ROI snapshot (animal {animalID}, pull {pullid_tgt})", fontsize=14)
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)

        print("Saved vector-overlay multi-page PDF:", pdf_name)

        # Optional: also save 3 individual PDFs (FIXED extension handling)
        for t_snap in sorted(snapshots.keys()):
            S = snapshots[t_snap]
            crop_rgb = cv2.cvtColor(S["crop_bgr"], cv2.COLOR_BGR2RGB)
            mag_u8 = (np.clip(S["mag_norm"], 0, 1) * 255).astype(np.uint8)

            fig = plt.figure(figsize=(14, 5.2))
            gs = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.0, 1.2], wspace=0.02)

            ax1 = fig.add_subplot(gs[0,0]); ax2 = fig.add_subplot(gs[0,1]); ax3 = fig.add_subplot(gs[0,2])

            ax1.imshow(crop_rgb); ax1.axis("off")
            add_roi_rect(ax1, S["roi_boxes"]["face"],  "white", "face")
            add_roi_rect(ax1, S["roi_boxes"]["left"],  "red",   "L")
            add_roi_rect(ax1, S["roi_boxes"]["right"], "blue",  "R")
            add_roi_rect(ax1, S["roi_boxes"]["under"], "lime",  "U")
            ax1.set_title(f"Crop (t={t_snap:+.1f}s)", fontsize=12)

            im = ax2.imshow(
                S["mag_norm"],          # <-- use the FLOAT data, not uint8
                cmap="gray",
                vmin=0,
                vmax=1
            )
            ax2.axis("off")

            add_roi_rect(ax2, S["roi_boxes"]["face"],  "white")
            add_roi_rect(ax2, S["roi_boxes"]["left"],  "red")
            add_roi_rect(ax2, S["roi_boxes"]["right"], "blue")
            add_roi_rect(ax2, S["roi_boxes"]["under"], "lime")

            ax2.set_title("Optic-flow magnitude", fontsize=12)

            # ---- ADD COLORBAR HERE ----
            cbar = fig.colorbar(
                im,
                ax=ax2,
                fraction=0.046,
                pad=0.04
            )
            cbar.set_label("Optic-flow magnitude (normalized, a.u.)", fontsize=10)


            ax3.plot(S["t_arr"], S["left"],  color="red",  label="L", linewidth=2)
            ax3.plot(S["t_arr"], S["right"], color="blue", label="R", linewidth=2)
            ax3.plot(S["t_arr"], S["under"], color="green",label="U", linewidth=2)
            ax3.plot(S["t_arr"], S["face"],  color="black",label="face", linewidth=2.5)
            ax3.axvline(0, color="gold", linestyle="--", linewidth=2, label="pull")
            ax3.axvline(S["t_now"], color="cyan", linestyle="-", linewidth=1)
            ax3.set_xlim(-4, 4)
            ax3.set_xlabel("Time from pull (s)")
            ax3.set_ylabel("Mean motion (norm.)")
            ax3.legend(frameon=False, fontsize=10, loc="upper right")
            ax3.set_title("ROI mean motion traces", fontsize=12)

            fig.suptitle(f"animal {animalID}, pull {pullid_tgt}, t={t_snap:+.1f}s", fontsize=14)

            # --- FIX: sanitize ONLY the time part, then append .pdf ---
            t_tag = f"{t_snap:+.1f}s"              # e.g. "+0.0s", "-1.0s"
            t_tag = t_tag.replace("+", "p").replace("-", "m").replace(".", "p")  # -> p0p0s, m1p0s

            fname = f"opticflow_vector_snapshot_{date_tgt}_animal{animalID}_pull{pullid_tgt}_t{t_tag}.pdf"
            plt.savefig(fname, bbox_inches="tight")  # extension stays .pdf so it saves as PDF
            plt.close(fig)
            print("Saved:", fname)

    else:
        print("No snapshots captured  check snapshot indices / fps / window.")

    # -----------------------------
    # Convert AVI -> MP4 for Mac (if needed)
    # -----------------------------
    if video_out is not None and video_out.lower().endswith(".avi"):
        avi_path = video_out
        mp4_path = avi_path[:-4] + ".mp4"

        cmds = [
            ["ffmpeg", "-y", "-i", avi_path, "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "23", "-preset", "medium", mp4_path],
            ["ffmpeg", "-y", "-i", avi_path, "-c:v", "mpeg4", "-q:v", "5", "-pix_fmt", "yuv420p", mp4_path],
        ]

        for cmd in cmds:
            try:
                print("Running:", " ".join(cmd))
                subprocess.run(cmd, check=True)
                print("Converted to:", mp4_path)
                break
            except Exception as e:
                print("Conversion attempt failed:", e)


In [ ]:
# ===== Summary optic-flow motion plot across ALL pulls (4 panels: animal1/2 x raw/z) + PDF save =====
# Self-contained (does NOT require you to run the demo-video cell first), but assumes these variables already exist:
#   video_file_original, session_start_time, time_point_pull1, time_point_pull2, bodyparts_locs_camI, date_tgt
#
# Output:
#   - Displays a 2x2 figure in notebook
#   - Saves a PDF in the current working directory

if 0:

    import numpy as np
    import cv2
    import matplotlib.pyplot as plt

    # -----------------------------
    # USER PARAMETERS
    # -----------------------------
    fps = 30
    win_pre = 4.0
    win_post = 4.0
    mag_clip = 8.0                 # normalize magnitude by this (for raw plot). Increase -> smaller raw values.
    sqr_thres_face = 1.30          # make face ROI a bit bigger
    xpoint_crop = 950              # half-screen crop boundary
    baseline_win = (-4.0, -2.0)    # for z-scoring (seconds relative to pull)
    max_ffill_s = 0.5              # forward-fill tracking dropouts up to this long (seconds); beyond -> NaNs

    # Farneback params
    fb_params = dict(
        pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0
    )

    # Animal mapping (tracking identity)
    animals = {
        1: dict(name="dodson", pull_times=time_point_pull1),
        2: dict(name="scorch", pull_times=time_point_pull2),
    }

    video_in = video_file_original
    bp = bodyparts_locs_camI

    # -----------------------------
    # Helper functions
    # -----------------------------
    def clip_box_xyxy(x0, x1, y0, y1, w, h):
        x0 = int(max(0, min(w-1, x0)))
        x1 = int(max(1, min(w,   x1)))
        y0 = int(max(0, min(h-1, y0)))
        y1 = int(max(1, min(h,   y1)))
        if (x1-x0) < 2 or (y1-y0) < 2:
            return None
        return (x0, x1, y0, y1)

    def mean_in_box(mag_map, box):
        if box is None:
            return np.nan
        x0,x1,y0,y1 = box
        patch = mag_map[y0:y1, x0:x1]
        if patch.size < 4:
            return np.nan
        return float(np.nanmean(patch))

    def safe_xy(bp_dict, key, i):
        """Return (x,y) from bp_dict[key][i,:] or (nan,nan) if missing/out-of-range."""
        arr = bp_dict.get(key, None)
        if arr is None:
            return np.array([np.nan, np.nan], dtype=float)
        try:
            if i < 0 or i >= arr.shape[0]:
                return np.array([np.nan, np.nan], dtype=float)
            xy = np.array(arr[i, :], dtype=float)
            if xy.shape[0] != 2:
                return np.array([np.nan, np.nan], dtype=float)
            return xy
        except Exception:
            return np.array([np.nan, np.nan], dtype=float)

    def flow_mag_norm(prev_gray, gray, mag_clip=8.0):
        """Compute Farneback flow magnitude (normalized to 0..1 via mag_clip)."""
        flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, **fb_params)
        fx, fy = flow[..., 0], flow[..., 1]
        mag = np.sqrt(fx*fx + fy*fy)
        return np.clip(mag / mag_clip, 0, 1)

    def zscore_trials(X, t, baseline=(-4.0, -2.0)):
        """Z-score each trial by its own baseline window."""
        t0, t1 = baseline
        base_mask = (t >= t0) & (t <= t1)
        # if baseline mask empty (shouldn't happen), return NaNs
        if not np.any(base_mask):
            return np.full_like(X, np.nan)
        mu = np.nanmean(X[:, base_mask], axis=1, keepdims=True)
        sd = np.nanstd(X[:, base_mask], axis=1, keepdims=True)
        sd = np.where(sd < 1e-6, np.nan, sd)
        return (X - mu) / sd

    def mean_sem(X):
        """Mean and SEM across trials (ignore NaNs)."""
        mean = np.nanmean(X, axis=0)
        # count finite per time
        n = np.sum(np.isfinite(X), axis=0)
        sd = np.nanstd(X, axis=0)
        sem = sd / np.sqrt(np.maximum(n, 1))
        return mean, sem

    # -----------------------------
    # Core extraction across pulls (one session)
    # -----------------------------
    def extract_motion_traces(animalID):
        """
        Returns:
          tvec (nT,)
          face, left, right, under arrays each shape (nTrials, nT)
        Using forward-fill for short tracking dropouts so traces always have length nT.
        """
        name = animals[animalID]["name"]
        pulls = np.array(animals[animalID]["pull_times"], dtype=float)

        # Build time vector and frame count (we compute flow between consecutive frames, so length = nframes)
        nT = int((win_pre + win_post) * fps)
        tvec = np.arange(-win_pre, win_post, 1/fps)  # length nT
        assert len(tvec) == nT

        cap = cv2.VideoCapture(video_in)
        if not cap.isOpened():
            raise RuntimeError(f"Could not open video: {video_in}")

        W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # Crop bounds
        if animalID == 1:
            X0, X1 = int(xpoint_crop), W
        else:
            X0, X1 = 0, int(xpoint_crop)
        crop_w = X1 - X0

        # Heuristic to decide if tracking indices are session-relative or absolute
        # If absolute frame numbers exceed tracking length, use session-relative index.
        arr_test = bp.get((name, "rightTuft"), None)
        if arr_test is None:
            raise KeyError(f"bodyparts_locs_camI missing key: ({name}, 'rightTuft')")
        tracking_len = arr_test.shape[0]

        frame0_session = int(np.round(session_start_time * fps))
        # We'll decide per pull, but typically it's consistent:
        # absolute indices are ~1e4; tracking_len usually ~video length in session coords.
        # If frame0_session is big and tracking_len is much smaller, then tracking is session-relative.
        # We'll compute itrack = iframe - frame0_session when iframe>=tracking_len.
        def track_index(iframe):
            return (iframe - frame0_session) if iframe >= tracking_len else iframe

        max_ffill_frames = int(np.round(max_ffill_s * fps))

        all_face, all_left, all_right, all_under = [], [], [], []

        for p in pulls:
            # absolute iframe range for the video
            iframe_start = int(np.round(frame0_session + (p - win_pre) * fps))
            iframe_end   = int(np.round(iframe_start + nT))  # we will read nT frames after first

            # Guard: skip if before 0
            if iframe_start < 1:
                continue

            cap.set(cv2.CAP_PROP_POS_FRAMES, iframe_start)
            ok, frame0 = cap.read()
            if not ok:
                continue

            prev_gray = cv2.cvtColor(frame0[:, X0:X1], cv2.COLOR_BGR2GRAY)

            face_tr, left_tr, right_tr, under_tr = [], [], [], []

            # forward-fill state (last valid boxes)
            last_face_box = last_left_box = last_right_box = last_under_box = None
            miss_count = 0

            # iterate exactly nT frames (each step appends exactly one value)
            for k in range(nT):
                # read next frame
                ok, frame = cap.read()
                if not ok:
                    # pad remaining with NaNs to keep length consistent
                    remaining = nT - k
                    face_tr.extend([np.nan] * remaining)
                    left_tr.extend([np.nan] * remaining)
                    right_tr.extend([np.nan] * remaining)
                    under_tr.extend([np.nan] * remaining)
                    break

                iframe = iframe_start + 1 + k
                gray = cv2.cvtColor(frame[:, X0:X1], cv2.COLOR_BGR2GRAY)

                mag = flow_mag_norm(prev_gray, gray, mag_clip=mag_clip)
                prev_gray = gray

                # tracking index
                itrack = track_index(iframe)

                # get landmarks (session-relative or absolute depending on track_index)
                rightTuft  = safe_xy(bp, (name, "rightTuft"), itrack)
                leftTuft   = safe_xy(bp, (name, "leftTuft"), itrack)
                whiteBlaze = safe_xy(bp, (name, "whiteBlaze"), itrack)
                rightEye   = safe_xy(bp, (name, "rightEye"), itrack)
                leftEye    = safe_xy(bp, (name, "leftEye"), itrack)
                mouth      = safe_xy(bp, (name, "mouth"), itrack)

                pts = np.vstack([rightTuft, whiteBlaze, leftTuft, leftEye, mouth, rightEye])
                face_mass = np.nanmean(pts, axis=0)

                # robust face size estimate: use max distance to face_mass
                d = np.nanmax(np.linalg.norm(pts - face_mass, axis=1))

                # If tracking valid, update boxes; else forward-fill last boxes (limited by max_ffill_frames)
                if np.isfinite(d) and d > 1 and np.all(np.isfinite(face_mass)):
                    miss_count = 0

                    face_offset = d * sqr_thres_face
                    fx = face_mass[0] - X0
                    fy = face_mass[1]

                    # face box
                    fx0, fx1 = int(fx - face_offset), int(fx + face_offset)
                    fy0, fy1 = int(fy - face_offset), int(fy + face_offset)
                    fw, fh = fx1 - fx0, fy1 - fy0

                    face_box = clip_box_xyxy(fx0, fx1, fy0, fy1, crop_w, H)

                    # requested geometry:
                    # left/right: same height as face box (fh), width = fw
                    left_box  = clip_box_xyxy(fx0 - fw, fx0, fy0, fy0 + fh, crop_w, H)
                    right_box = clip_box_xyxy(fx1, fx1 + fw, fy0, fy0 + fh, crop_w, H)
                    # under: same width as face, height = fh (one face-box height under)
                    under_box = clip_box_xyxy(fx0, fx1, fy1, fy1 + fh, crop_w, H)

                    if face_box is not None:
                        last_face_box  = face_box
                        last_left_box  = left_box
                        last_right_box = right_box
                        last_under_box = under_box
                else:
                    miss_count += 1
                    if miss_count > max_ffill_frames:
                        last_face_box = last_left_box = last_right_box = last_under_box = None

                # append using current (possibly forward-filled) boxes
                face_tr.append(mean_in_box(mag, last_face_box))
                left_tr.append(mean_in_box(mag, last_left_box))
                right_tr.append(mean_in_box(mag, last_right_box))
                under_tr.append(mean_in_box(mag, last_under_box))

            # keep only if correct length
            if len(face_tr) == nT:
                all_face.append(face_tr)
                all_left.append(left_tr)
                all_right.append(right_tr)
                all_under.append(under_tr)

        cap.release()

        # Convert to arrays (nTrials, nT)
        face = np.array(all_face, dtype=float)
        left = np.array(all_left, dtype=float)
        right = np.array(all_right, dtype=float)
        under = np.array(all_under, dtype=float)

        return tvec, face, left, right, under

    # -----------------------------
    # Run + plot (2x2)
    # -----------------------------
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)

    for col, animalID in enumerate([1, 2]):
        t, face, L, R, U = extract_motion_traces(animalID)
        ntr = face.shape[0]
        print(f"Animal {animalID}: extracted {ntr} pull windows")

        # RAW
        face_m, face_s = mean_sem(face)
        L_m, L_s       = mean_sem(L)
        R_m, R_s       = mean_sem(R)
        U_m, U_s       = mean_sem(U)

        ax = axes[0, col]
        ax.axvline(0, linestyle="--", linewidth=1)
        ax.plot(t, face_m, label="face")
        ax.fill_between(t, face_m-face_s, face_m+face_s, alpha=0.2)

        ax.plot(t, L_m, label="left")
        ax.fill_between(t, L_m-L_s, L_m+L_s, alpha=0.2)

        ax.plot(t, R_m, label="right")
        ax.fill_between(t, R_m-R_s, R_m+R_s, alpha=0.2)

        ax.plot(t, U_m, label="under")
        ax.fill_between(t, U_m-U_s, U_m+U_s, alpha=0.2)

        ax.set_title(f"Animal {animalID} raw motion (mean±SEM, n={ntr})")
        ax.set_ylabel("motion (normalized magnitude)")

        # Z-SCORED
        face_z = zscore_trials(face, t, baseline=baseline_win)
        L_z    = zscore_trials(L, t, baseline=baseline_win)
        R_z    = zscore_trials(R, t, baseline=baseline_win)
        U_z    = zscore_trials(U, t, baseline=baseline_win)

        face_mz, face_sz = mean_sem(face_z)
        L_mz, L_sz       = mean_sem(L_z)
        R_mz, R_sz       = mean_sem(R_z)
        U_mz, U_sz       = mean_sem(U_z)

        ax = axes[1, col]
        ax.axvline(0, linestyle="--", linewidth=1)
        ax.plot(t, face_mz, label="face")
        ax.fill_between(t, face_mz-face_sz, face_mz+face_sz, alpha=0.2)

        ax.plot(t, L_mz, label="left")
        ax.fill_between(t, L_mz-L_sz, L_mz+L_sz, alpha=0.2)

        ax.plot(t, R_mz, label="right")
        ax.fill_between(t, R_mz-R_sz, R_mz+R_sz, alpha=0.2)

        ax.plot(t, U_mz, label="under")
        ax.fill_between(t, U_mz-U_sz, U_mz+U_sz, alpha=0.2)

        ax.set_title(f"Animal {animalID} z-scored (baseline {baseline_win[0]}..{baseline_win[1]} s)")
        ax.set_ylabel("z-scored motion")

    # Legend on top-left panel
    axes[0,0].legend(frameon=False, fontsize=10)
    for ax in axes[1,:]:
        ax.set_xlabel("Time from pull (s)")

    plt.tight_layout()

    # Save figure as PDF in current directory
    fig_name = f"opticflow_face_vs_body_summary_{date_tgt}.pdf"
    plt.savefig(fig_name, bbox_inches="tight")
    print(f"Saved figure to: {fig_name}")

    plt.show()
